In [1]:
import pandas as pd
import re
from sklearn.metrics import classification_report, confusion_matrix

def get_infer(path, test):
    if test:
        start = 226 
        mid = 452
    else:
        start = 203
        mid = 406
    dataset = pd.read_csv(path)
    dataset = dataset.drop(columns = ["Unnamed: 0"])
    print("Longitud del dataset: {}".format(len(dataset)))
    infer = dataset[start:mid]
    return infer

llm_test_ds = get_infer("/home/flopezp/Kurosagol/Ongoing/baseline_datasets/test/DeepSeek-R1-0528-Qwen3-8B_full.csv", True)
llm_dev_ds = get_infer("/home/flopezp/Kurosagol/Ongoing/baseline_datasets/validation/DeepSeek-R1-0528-Qwen3-8B_full.csv", False)

folio_test = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
test_values = [1 if folio_test["label"].iloc[i] == 'True' else 0 if folio_test["label"].iloc[i] == 'False' else 2 for i in range(len(folio_test))]

folio_val = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)
val_values = [1 if folio_val["label"].iloc[i] == 'True' else 0 if folio_val["label"].iloc[i] == 'False' else 2 for i in range(len(folio_val))]

Longitud del dataset: 678
Longitud del dataset: 609


In [4]:
def get_labels(dataset, test):
    values = []
    for i in range(len(dataset)):
        text = dataset["Full"].iloc[i]
        text = re.sub('\n', '', text)
        unc = (text.count('Uncertain') + text.count('uncertain'), 'Uncertain')
        false = (text.count('False') + text.count('false'), 'False')
        true = (text.count('True') + text.count('true'), 'True')
        biggest = max(unc, false, true, key=lambda x:x[0])
        #print(f"Label: {biggest[1]}. T = {true[0]}, F= {false[0]}, U = {unc[0]}")
        if biggest[1] == 'True':
            values.append(1)
        elif biggest[1] == 'False':
            values.append(0)
        else:
            values.append(2)

    if test:
        print("------------------")
        print("====== TEST ======")
        print("------------------")
        print(classification_report(values, test_values))
        print(confusion_matrix(values, test_values))
    else:
        print("-----------------")
        print("====== VAL ======")
        print("-----------------")
        print(classification_report(values, val_values))
        print(confusion_matrix(values, val_values))

get_labels(llm_test_ds, True)
get_labels(llm_dev_ds, False)

------------------
====== TEST ======
------------------
              precision    recall  f1-score   support

           0       0.34      0.46      0.39        46
           1       0.87      0.48      0.62       158
           2       0.19      0.68      0.30        22

    accuracy                           0.50       226
   macro avg       0.47      0.54      0.44       226
weighted avg       0.70      0.50      0.54       226

[[21  8 17]
 [36 76 46]
 [ 4  3 15]]
-----------------
====== VAL ======
-----------------
              precision    recall  f1-score   support

           0       0.47      0.59      0.52        49
           1       0.83      0.48      0.61       126
           2       0.28      0.68      0.39        28

    accuracy                           0.53       203
   macro avg       0.53      0.58      0.51       203
weighted avg       0.67      0.53      0.56       203

[[29  7 13]
 [29 60 37]
 [ 4  5 19]]
